# GPU Regime Alpha — Full Pipeline Walkthrough

End-to-end zero-copy GPU market regime classification, from raw order book
ticks to regime-conditioned alpha prediction. Every step below runs on a
single NVIDIA GPU via RAPIDS (cuDF/cuML/CuPy) and cuVS.

See `article/narrative_draft.md` for full technical writeup including
honest engineering failures and benchmark methodology.

In [1]:
import sys
sys.path.insert(0, '..')


## 1. Generate synthetic multi-regime LOB data

In [2]:
from data.synthetic.lob_generator import generate_synthetic_lob

df = generate_synthetic_lob(n_ticks=100_000)
df.head()

,timestamp,mid_price,best_bid,best_ask,bid_size,ask_size,trade_price,trade_size,order_flow_sign
0,0.001,99.999953,99.989427,100.010480,553.706955,577.533299,100.010480,117.524789,1.0
1,0.002,99.999855,99.989608,100.010102,525.346716,538.995753,100.010102,77.451337,1.0
2,0.003,99.999798,99.990264,100.009332,480.990629,336.480887,99.990264,87.420974,-1.0
3,0.004,99.999696,99.992914,100.006479,496.243181,276.425802,99.992914,23.963174,-1.0
4,0.005,99.999646,99.990927,100.008364,363.211003,521.827236,99.990927,146.958760,-1.0


## 2. Feature engineering — 20 GPU-native modules in VRAM

In [3]:
from fusion.feature_matrix_builder import build_full_feature_matrix
import time

start = time.perf_counter()
features = build_full_feature_matrix(
    df,
    rolling_window=100,
    tail_window=200,
    entropy_window=100,
    hurst_window=300,
    spectral_window=100,
)
print(f"feature engineering: {time.perf_counter() - start:.3f}s")
features.head()

feature engineering: 1.060s


,ofi,spread,relative_spread,micro_price,micro_price_var,log_return,realized_var,realized_vol,hill_tail_index,permutation_entropy,hurst_exponent,mp_deviation,max_eigenvalue,spectral_gap,eigenvector_stability,eigenvalue_entropy,timestamp
0,0.000000,0.021053,0.000211,99.999732,<NA>,0.000000e+00,0.000000e+00,0.000000e+00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.001
1,-13.649037,0.020495,0.000205,99.999724,3.333676159e-11,-9.846509e-07,9.695373e-13,9.846509e-07,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.002
2,144.509742,0.019068,0.000191,100.001483,1.027360007e-06,-5.716966e-07,1.296374e-12,1.138584e-06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.003
3,219.817379,0.013565,0.000136,100.001626,1.115998363e-06,-1.013370e-06,2.323292e-12,1.524235e-06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.004
4,0.000000,0.017437,0.000174,99.998083,2.145599307e-06,-5.070304e-07,2.580372e-12,1.606354e-06,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.005


Feature set spans market microstructure (OFI, spread, micro-price,
realized volatility, tail index), nonlinear dynamics (permutation entropy,
Hurst/DFA), and spectral/RMT (rolling correlation eigenstructure).

## 3. Topology — Vietoris-Rips persistence on phase-space embedding

In [4]:
from features.microstructure.realized_vol import compute_log_returns
from features.topology.vietoris_rips import compute_persistence_diagrams_from_series, diagram_to_cupy
from features.topology.wasserstein_pd import sinkhorn_divergence_diagrams

log_returns = compute_log_returns(df).to_cupy()

window_a = log_returns[:500]
window_b = log_returns[500:1000]

dgms_a = compute_persistence_diagrams_from_series(window_a, embedding_dim=3, delay=1, maxdim=1)
dgms_b = compute_persistence_diagrams_from_series(window_b, embedding_dim=3, delay=1, maxdim=1)

h1_a = diagram_to_cupy(dgms_a[1])
h1_b = diagram_to_cupy(dgms_b[1])

shift_score = sinkhorn_divergence_diagrams(h1_a, h1_b)
self_distance = sinkhorn_divergence_diagrams(h1_a, h1_a)

print(f"structural shift score (different windows): {shift_score:.4e}")
print(f"self-distance sanity check (should be ~0): {self_distance:.4e}")

structural shift score (different windows): 1.4487e-08
self-distance sanity check (should be ~0): 0.0000e+00


In [5]:
from viz.persistence_diagram_plot import build_persistence_diagram_figure

fig = build_persistence_diagram_figure(h1_a, title="Persistence Diagram (H1) — window A")
fig.show()

## 4. UMAP reduction + HDBSCAN unsupervised regime discovery

In [6]:
from reduction.umap_reduce import fit_umap_reduction
from reduction.hdbscan_cluster import fit_hdbscan_clusters, cluster_summary, attach_cluster_labels

feature_cols = [c for c in features.columns if c != "timestamp"]
embedding, clean_features = fit_umap_reduction(features, feature_cols, n_components=3)

labels, clusterer = fit_hdbscan_clusters(embedding, min_cluster_size=100)
print(cluster_summary(labels))

clustered = attach_cluster_labels(clean_features, labels)

[2026-08-20 23:37:29.608] [CUML] [info] build_algo set to brute_force_knn because random_state is given


{'n_clusters': 11, 'n_noise_points': 445, 'noise_fraction': 0.004463345402754235, 'cluster_sizes': {0: 3252, 1: 189, 2: 577, 3: 4616, 4: 85691, 5: 236, 6: 826, 7: 552, 8: 1149, 9: 1515, 10: 653}}


In [7]:
from viz.umap_3d_plot import build_umap_3d_figure

fig = build_umap_3d_figure(embedding, labels, title="LOB Regime Clusters")
fig.show()

In [8]:
from viz.regime_timeline_plot import build_regime_timeline_figure

timestamps = clean_features["timestamp"].to_cupy()
prices = clean_features["micro_price"].to_cupy()

fig = build_regime_timeline_figure(timestamps, prices, labels, title="Regime Timeline")
fig.show()

## 5. cuVS CAGRA — microsecond live regime matching

In [9]:
from retrieval.cagra_index_builder import build_cagra_index, query_nearest_regime
import time

index = build_cagra_index(embedding)

query_points = embedding[:100]
start = time.perf_counter()
matched_labels = query_nearest_regime(index, query_points, labels)
elapsed = time.perf_counter() - start

print(f"CAGRA query for 100 points: {elapsed*1e6:.1f} microseconds total")
print(f"self-match rate (sanity check): {float((matched_labels == labels[:100]).mean()):.2%}")

[112980][23:37:43:206965][info  ] optimizing graph
[112980][23:37:43:383115][info  ] Graph optimized, creating index


CAGRA query for 100 points: 115629.7 microseconds total


self-match rate (sanity check): 100.00%


## 6. Regime-conditioned XGBoost alpha prediction

In [10]:
from model.xgb_conditioned_alpha import build_training_frame, walk_forward_split, train_conditioned_xgb, predict_alpha
import cupy as cp

model_feature_cols = feature_cols
training_frame = build_training_frame(clustered, df, horizon=10, feature_columns=model_feature_cols)
train, test = walk_forward_split(training_frame, train_fraction=0.7)

booster = train_conditioned_xgb(train, model_feature_cols, num_boost_round=100)
predictions = predict_alpha(booster, test, model_feature_cols)
actual = test["target"].to_cupy()

mse = float(cp.mean((predictions - actual) ** 2))
baseline_mse = float(cp.mean((actual - cp.mean(actual)) ** 2))

print(f"test MSE: {mse:.6f}")
print(f"baseline (mean predictor) MSE: {baseline_mse:.6f}")
print(f"improvement over baseline: {1 - mse/baseline_mse:.4%}")

test MSE: 0.035476
baseline (mean predictor) MSE: 0.019163
improvement over baseline: -85.1251%


On purely synthetic Gaussian random-walk data, no improvement over baseline
is the *correct* result — there is no real alpha signal in pure noise. This
notebook demonstrates the infrastructure and mathematics, not a trading
strategy. See `article/narrative_draft.md` for the full discussion.

## 7. End-to-end benchmark: CPU vs GPU

In [11]:
from benchmark.gpu_pipeline_bench import run_full_pipeline_benchmark

print("=== warm-up run (JIT compilation) ===")
run_full_pipeline_benchmark(n_ticks=5_000)

print()
print("=== measured run ===")
timings = run_full_pipeline_benchmark(n_ticks=100_000)
timings

=== warm-up run (JIT compilation) ===
[data_generation] 0.0074s
[feature_engineering] 0.1166s
[2026-08-20 23:37:47.716] [CUML] [info] build_algo set to brute_force_knn because random_state is given
[umap_reduction] 0.0425s


[hdbscan_clustering] 0.1024s
[cagra_index_build] 0.2110s


[112980][23:37:48:040335][info  ] optimizing graph
[112980][23:37:48:064068][info  ] Graph optimized, creating index


[cagra_query] 0.0030s
[training_frame_build] 0.0133s
[walk_forward_split] 0.0014s


[xgboost_train] 0.3253s
[xgboost_predict] 0.0158s

[TOTAL] 0.8387s for n_ticks=5000

=== measured run ===
[data_generation] 0.0087s


...

[hdbscan_clustering] 3.0422s
[cagra_index_build] 0.5024s


[112980][23:37:52:903534][info  ] optimizing graph
[112980][23:37:53:074051][info  ] Graph optimized, creating index


[cagra_query] 0.0031s
[training_frame_build] 0.0164s
[walk_forward_split] 0.0013s


[xgboost_train] 0.4824s
[xgboost_predict] 0.0165s

[TOTAL] 5.1716s for n_ticks=100000


{'data_generation': 0.008749698999963584,
 'feature_engineering': 0.8775329580003017,
 'umap_reduction': 0.22092635599983623,
 'hdbscan_clustering': 3.042248054999618,
 'cagra_index_build': 0.5023577419997309,
 'cagra_query': 0.0031409949997396325,
 'training_frame_build': 0.016415891999713494,
 'walk_forward_split': 0.0013083690000712522,
 'xgboost_train': 0.4824384789999385,
 'xgboost_predict': 0.016499536999617703,
 'total': 5.171618081998531}

Full CPU vs GPU comparison across multiple scales (5K / 100K / 500K ticks),
including per-stage breakdown and verified GPU utilization logs, is in
`benchmark_report.md` and `article/narrative_draft.md`.